In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,to_date,datediff,when

spark=SparkSession.builder.appName("etl").getOrCreate()

df=spark.read.csv(
'orders.csv',
header=True,
inferSchema=True
)

df.show()

+--------+-----------+-------------+---------+
|order_id|supplier_id|delivery_date|   status|
+--------+-----------+-------------+---------+
|     101|          1|   2026-05-01|delivered|
|     102|          2|   2026-05-12|  delayed|
|     103|          1|   2026-05-08|delivered|
|     104|          2|   2026-05-03|  delayed|
+--------+-----------+-------------+---------+



In [4]:
df_clean=df.withColumn(
'delivery_date',
to_date(col('delivery_date'),'yyyy-MM-dd'))
df_clean=df_clean.withColumn(
'delay_days',
datediff(col('delivery_date'),to_date(col('delivery_date'))))
df_clean=df_clean.withColumn(
'is_delayed',
when(col('status')=='delayed',1).otherwise(0)
)
df_clean.show()

+--------+-----------+-------------+---------+----------+----------+
|order_id|supplier_id|delivery_date|   status|delay_days|is_delayed|
+--------+-----------+-------------+---------+----------+----------+
|     101|          1|   2026-05-01|delivered|         0|         0|
|     102|          2|   2026-05-12|  delayed|         0|         1|
|     103|          1|   2026-05-08|delivered|         0|         0|
|     104|          2|   2026-05-03|  delayed|         0|         1|
+--------+-----------+-------------+---------+----------+----------+



In [5]:
df_delayed=df_clean.filter(
col('status')=='delayed'
)

df_delayed.show()

+--------+-----------+-------------+-------+----------+----------+
|order_id|supplier_id|delivery_date| status|delay_days|is_delayed|
+--------+-----------+-------------+-------+----------+----------+
|     102|          2|   2026-05-12|delayed|         0|         1|
|     104|          2|   2026-05-03|delayed|         0|         1|
+--------+-----------+-------------+-------+----------+----------+



In [6]:
df_delayed.toPandas().to_csv(
'cleaned_output.csv',
index=False
)